In [1]:
import pandas as pd
import json
import os

print("Biblioteker lastet inn!")


Biblioteker lastet inn!


In [3]:
DATA_FILE = "../data/filtertreff_ukodede_diagnoser.csv"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE, encoding="utf-8", sep=";")
    df = df.rename(columns={
        "journal_id": "id",
        "diagnosetekst": "diagnosis"
    })
    print(f"Lastet {len(df)} rader fra fil.")
else:
    print("Fant ikke filen! Sjekk at den ligger i data/-mappen.")

df.head()

Lastet 4388 rader fra fil.


,id,diagnosis
0,04DD8FF1-66C0-453E-BFCF-A576624862E6,hysteria ?
1,10478F81-98D2-4F83-8EA6-1EDF1811680C,hysteria 7
2,97883cca-48aa-46f1-bab0-57f234b2f598,hysteria
3,E8F128A2-502F-47E8-B73F-6566E52D4B95,202|hysteria
4,8F1CC767-D817-4334-B717-D493D2EBC80B,hysteria


In [4]:
print(f"Antall rader: {df.shape[0]}")
print(f"Antall kolonner: {df.shape[1]}")
print(f"\nManglende verdier:\n{df.isnull().sum()}")
print(f"\nDe første radene:")
df.head()

Antall rader: 4388
Antall kolonner: 2

Manglende verdier:
id           0
diagnosis    0
dtype: int64

De første radene:


,id,diagnosis
0,04DD8FF1-66C0-453E-BFCF-A576624862E6,hysteria ?
1,10478F81-98D2-4F83-8EA6-1EDF1811680C,hysteria 7
2,97883cca-48aa-46f1-bab0-57f234b2f598,hysteria
3,E8F128A2-502F-47E8-B73F-6566E52D4B95,202|hysteria
4,8F1CC767-D817-4334-B717-D493D2EBC80B,hysteria


In [5]:
# Fjern tomme diagnoser
df = df.dropna(subset=["diagnosis"])
df = df[df["diagnosis"].str.strip() != ""]

# Fjern ekstra mellomrom
df["diagnosis"] = df["diagnosis"].str.strip()
df["diagnosis"] = df["diagnosis"].str.replace(r"\s+", " ", regex=True)

# Lag en lowercase-versjon for sammenligning
df["diagnosis_clean"] = df["diagnosis"].str.lower()

# Fjern duplikater
before = len(df)
df = df.drop_duplicates(subset=["id", "diagnosis_clean"])
after = len(df)

print(f"Fjernet {before - after} tomme/duplikater.")
print(f"Gjenstående poster: {after}")
df

Fjernet 1285 tomme/duplikater.
Gjenstående poster: 3103


,id,diagnosis,diagnosis_clean
0,04DD8FF1-66C0-453E-BFCF-A576624862E6,hysteria ?,hysteria ?
1,10478F81-98D2-4F83-8EA6-1EDF1811680C,hysteria 7,hysteria 7
2,97883cca-48aa-46f1-bab0-57f234b2f598,hysteria,hysteria
3,E8F128A2-502F-47E8-B73F-6566E52D4B95,202|hysteria,202|hysteria
4,8F1CC767-D817-4334-B717-D493D2EBC80B,hysteria,hysteria
...,...,...,...
4383,3b619f2a-790c-4650-9dab-2085619eee89,psykoneurose hysteria,psykoneurose hysteria
4384,3b619f2a-790c-4650-9dab-2085619eee89,311 | psykoneurose hysteria,311 | psykoneurose hysteria
4385,67f5151c-fd3b-4c06-bd0d-46aad045f646,hysteria,hysteria
4386,ba0e5405-3223-4d8f-8b90-caea41b32424,hysterica,hysterica


In [6]:
OUTPUT_FILE = "../data/diagnoses_clean.json"

df.to_json(OUTPUT_FILE, orient="records", force_ascii=False, indent=2)

print(f"Lagret {len(df)} rensede poster til {OUTPUT_FILE}")

Lagret 3103 rensede poster til ../data/diagnoses_clean.json
